In [ ]:
# Set global font properties for all matplotlib plots
import matplotlib.pyplot as pl

# Configure global font settings
pl.rcParams['font.size'] = 17
pl.rcParams['axes.titlesize'] = 19
pl.rcParams['axes.labelsize'] = 17
pl.rcParams['xtick.labelsize'] = 16
pl.rcParams['ytick.labelsize'] = 16
pl.rcParams['legend.fontsize'] = 15
pl.rcParams['figure.titlesize'] = 21


# Optional: Change font family (uncomment if you want to change the font type)
# pl.rcParams['font.family'] = 'serif'  # Options: 'serif', 'sans-serif', 'monospace'
# pl.rcParams['font.serif'] = ['Times New Roman']  # Specific serif font
# pl.rcParams['font.sans-serif'] = ['Arial']  # Specific sans-serif font

print("Global font settings updated for all plots")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# Project root (run notebook from CHEOPSPR folder, or set this to the project path)
PROJECT_ROOT = os.getcwd()
os.chdir(PROJECT_ROOT)

# Planet to load: "HD202772Ab", "KELT7b", or "TOI1518b"
PLANET = "TOI1518b"

# Aperture to prefer (25 or 24); will use first match if exact not found
APERTURE = 25

COLORS = [
    "#773baf",  # rich purple
    "#74af3b",  # fresh green
    "#e51284",  # vivid pink
    "#B56576",  # dusty rose
    "#008080",  # deep teal
    "#e8d119",  # golden yellow
    "#C76B6B",  # muted red
    "#624D5B",  # aubergine grey
]

In [ ]:
def find_lightcurve_files(planet_name, aperture=None):
    """Find all *Lightcurve-R*_V0300.fits under planet folder (recursive)."""
    planet_dir = os.path.join(PROJECT_ROOT, planet_name)
    if not os.path.isdir(planet_dir):
        return []
    found = []
    for root, _, files in os.walk(planet_dir):
        for f in files:
            if "Lightcurve-R" in f and f.endswith("_V0300.fits"):
                path = os.path.join(root, f)
                # parse aperture from filename (e.g. Lightcurve-R25_V0300.fits)
                try:
                    r = int(f.split("Lightcurve-R")[1].split("_")[0])
                except Exception:
                    r = None
                found.append((path, r, root))
    # Prefer requested aperture, then sort by (aperture, path) for stable order
    if aperture is not None:
        preferred = [x for x in found if x[1] == aperture]
        if preferred:
            found = preferred
    found.sort(key=lambda x: (x[1] or 0, x[0]))
    return found


def load_one_lightcurve(filepath):
    """Load time, flux, flux_err and systematics from a single FITS lightcurve."""
    with fits.open(filepath) as hdul:
        data = hdul[1].data
        print(data.columns.names)  # Print available columns for debugging
        time = np.asarray(data["BJD_TIME"], dtype=float)
        flux = np.asarray(data["FLUX"], dtype=float)
        flux_err = np.asarray(data["FLUXERR"], dtype=float)
        roll_angle = np.asarray(data["ROLL_ANGLE"], dtype=float)
        centroid_x = np.asarray(data["CENTROID_X"], dtype=float)
        centroid_y = np.asarray(data["CENTROID_Y"], dtype=float)
        smearing_lc = np.asarray(data["SMEARING_LC"], dtype=float)
        background = np.asarray(data["BACKGROUND"], dtype=float)     
    med = np.nanmedian(flux)
    if med <= 0:
        med = 1.0
    flux_n = flux / med
    flux_err_n = flux_err / med
    return {
        "time": time,
        "flux": flux_n,
        "flux_err": flux_err_n,
        "roll_angle": roll_angle,
        "centroid_x": centroid_x,
        "centroid_y": centroid_y,
        "smearing_lc": smearing_lc,
        "background": background,
    }


def load_all_visits(planet_name, aperture=None):
    """Load all lightcurves for a planet; group by directory (one visit per dir)."""
    file_list = find_lightcurve_files(planet_name, aperture)
    if not file_list:
        return []
    # Group by directory so we get one "visit" per folder
    by_dir = {}
    for path, ap, root in file_list:
        if root not in by_dir:
            by_dir[root] = []
        by_dir[root].append((path, ap))
    visits = []
    for root in sorted(by_dir.keys()):
        # Use first file in folder (or prefer requested aperture)
        items = by_dir[root]
        if aperture is not None:
            best = [x for x in items if x[1] == aperture]
            if best:
                items = best
        path = items[0][0]
        visit_name = os.path.basename(root)
        d = load_one_lightcurve(path)
        visits.append({"name": visit_name, **d})
    return visits

In [ ]:
# Load all visits for the selected planet
visits = load_all_visits(PLANET, APERTURE)
print(f"Planet: {PLANET}")
print(f"Number of visits: {len(visits)}")
for v in visits:
    print(f"  - {v['name']}: {len(v['time'])} points")

In [ ]:
if not visits:
    print(f"No lightcurve FITS found under {PLANET}. Check path and that files are extracted.")
else:
    n_visits = len(visits)
    colors = [COLORS[i % len(COLORS)] for i in range(n_visits)]

    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    for i, v in enumerate(visits):
        ax.errorbar(
            v["time"],
            v["flux"],
            yerr=v["flux_err"],
            fmt=".",
            ms=4,
            alpha=0.85,
            color=colors[i],
            label=v["name"],
            capsize=1.5,
        )
    ax.set_xlabel("BJD (days)")
    ax.set_ylabel("Normalized flux")
    ax.set_title(f"{PLANET} — Light curves")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
    ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
    #ax.set_ylim(0.995, 1.005)
    fig.tight_layout()
    plt.show()

In [ ]:
# Now perform sigma clipping on flux and background, and re-plot using clipped data
if not visits:
    print(f"No lightcurve FITS found under {PLANET}. Check path and that files are extracted.")
else:
    # Check if flux has already been clipped to avoid re-clipping
    if not all('flux_clipped' in v for v in visits):
        # sigma-clipping parameters
        sigma = 5.0
        max_iters = 5

        # perform sigma clipping per visit and replace arrays with clipped versions
        clip_keys = [
            "time",
            "flux",
            "flux_err",
            "roll_angle",
            "centroid_x",
            "centroid_y",
            "smearing_lc",
            "background",
        ]
        for v in visits:
            f = np.asarray(v["flux"], dtype=float)
            good = np.isfinite(f) & np.isfinite(v["time"]) & np.isfinite(v["flux_err"])
            if not np.any(good):
                v["mask"] = good
                continue
            mask = good.copy()
            for _ in range(max_iters):
                if not np.any(mask):
                    break
                med = np.nanmedian(f[mask])
                std = np.nanstd(f[mask])
                new_mask = good & (np.abs(f - med) <= sigma * std)
                if new_mask.sum() == mask.sum():
                    mask = new_mask
                    break
                mask = new_mask
            # apply clipping to arrays
            for k in clip_keys:
                if k in v:
                    v[k] = np.asarray(v[k])[mask]
            # reset mask to all-True for the new (clipped) arrays
            v["mask"] = np.ones_like(v["flux"], dtype=bool)
        
        # Mark flux as clipped
        for v in visits:
            v['flux_clipped'] = True

    # Check if background has already been clipped to avoid re-clipping
    if not all('bg_clipped' in v for v in visits):
        # Sigma clip data points with high background values (above 5 sigma)
        sigma_bg = 5.0
        max_iters_bg = 5

        for v in visits:
            bg = np.asarray(v["background"], dtype=float)
            good = np.isfinite(bg)
            if not np.any(good):
                v["mask_bg"] = good
                continue
            mask = good.copy()
            for _ in range(max_iters_bg):
                if not np.any(mask):
                    break
                med = np.nanmedian(bg[mask])
                std = np.nanstd(bg[mask])
                new_mask = good & (bg <= med + sigma_bg * std)
                if new_mask.sum() == mask.sum():
                    mask = new_mask
                    break
                mask = new_mask
            # Apply clipping to all arrays (replace with clipped versions)
            for k in clip_keys:
                if k in v:
                    v[k] = np.asarray(v[k])[mask]
            # Reset mask to all-True for the new (clipped) arrays
            v["mask"] = np.ones_like(v["flux"], dtype=bool)
        
        # Mark background as clipped
        for v in visits:
            v['bg_clipped'] = True

    # plotting using clipped data
    n_visits = len(visits)
    colors = [COLORS[i % len(COLORS)] for i in range(n_visits)]

    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    for i, v in enumerate(visits):
        m = v.get("mask", np.ones_like(v["flux"], dtype=bool))
        ax.errorbar(
            v["time"][m],
            v["flux"][m],
            yerr=v["flux_err"][m],
            fmt=".",
            ms=4,
            alpha=0.85,
            color=colors[i],
            label=v["name"],
            capsize=1.5,
        )
    ax.set_xlabel("BJD (days)")
    ax.set_ylabel("Normalized flux")
    ax.set_title(f"{PLANET} — Light curves (sigma-clipped)")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
    ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
    #ax.set_ylim(0.995, 1.005)
    fig.tight_layout()
    plt.show()


### One subplot per visit: light curve + systematics (stacked, semi-transparent)

In [ ]:
if visits:
    from matplotlib.transforms import blended_transform_factory
    n = len(visits)
    ncol = min(1, n)
    nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(10 * ncol, 6 * nrow), squeeze=False)
    axes = axes.flatten()
    def _norm(y):
        y = np.nan_to_num(y, nan=np.nanmedian(y))
        r = np.nanmax(y) - np.nanmin(y)
        if r <= 0:
            return np.zeros_like(y)
        return (y - np.nanmin(y)) / r
    sys_alpha = 0.5
    band = 0.10
    offsets = [0.8, 0.6, 0.4, 0.2, 0.0]  # vertical offsets for systematics
    sys_names = ["Roll angle\n(degrees)", "Centroid X\n(pixels)", "Centroid Y\n(pixels)", "Smearing LC\n(e-/pix/sec)", "Background\n(e-/pix/sec)"]
    for i, v in enumerate(visits):
        ax = axes[i]
        c = COLORS[i % len(COLORS)]
        t, f, fe = v["time"], v["flux"], v["flux_err"]
        ax.errorbar(t, f, yerr=fe, fmt=".", ms=3, alpha=0.9, color=c, capsize=1, label="flux")
        roll = np.asarray(v["roll_angle"])
        cx = np.asarray(v["centroid_x"])
        cy = np.asarray(v["centroid_y"])
        sm = np.asarray(v["smearing_lc"])
        bc = np.asarray(v["background"])
        y_roll = offsets[0] + _norm(roll) * band
        y_cx = offsets[1] + _norm(cx) * band
        y_cy = offsets[2] + _norm(cy) * band
        y_sm = offsets[3] + _norm(sm) * band
        y_bc = offsets[4] + _norm(bc) * band
        ax.plot(t, y_roll, ".", ms=2, alpha=sys_alpha, color=c, label="roll")
        ax.plot(t, y_cx, ".", ms=2, alpha=sys_alpha, color=c, label="centroid_x")
        ax.plot(t, y_cy, ".", ms=2, alpha=sys_alpha, color=c, label="centroid_y")
        ax.plot(t, y_sm, ".", ms=2, alpha=sys_alpha, color=c, label="smearing")
        ax.plot(t, y_bc, ".", ms=2, alpha=sys_alpha, color=c, label="background")
        ax.set_title(v["name"])
        ax.set_xlabel("BJD (days)")
        ax.set_ylabel("Normalized flux")
        #ax.set_ylim(-0.02, 1.08)
        trans = blended_transform_factory(ax.transAxes, ax.transData)
        for j, name in enumerate(sys_names):
            yc = offsets[j] + band / 2
            ax.text(-0.14, yc, name, transform=trans, va="center", ha="right", fontsize=13)
        ax.grid(True, alpha=0.3)
        ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
    for j in range(len(visits), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(f"{PLANET} — Light curves + systematics by visit")
    fig.tight_layout()
    plt.show()

In [ ]:
# Additional plots: flux vs each systematic for all visits

if visits:
    sys_keys = [
        ("roll_angle", "Roll angle (degrees)"),
        ("centroid_x", "Centroid X (pixels)"),
        ("centroid_y", "Centroid Y (pixels)"),
        ("smearing_lc", "Smearing LC (e-/pix/sec)"),
        ("background", "Background (e-/pix/sec)"),
    ]
    n_sys = len(sys_keys)
    n_vis = len(visits)
    fig, axes = plt.subplots(n_sys, n_vis, figsize=(7 * n_vis, 4.5 * n_sys), squeeze=False)
    for j, (sys_key, sys_label) in enumerate(sys_keys):
        for i, v in enumerate(visits):
            ax = axes[j, i]
            sys_vals = np.asarray(v[sys_key])
            flux = np.asarray(v["flux"])
            ax.plot(sys_vals, flux, ".", ms=3, alpha=0.7, color=COLORS[i % len(COLORS)])
            ax.set_xlabel(sys_label)
            if i == 0:
                ax.set_ylabel("Normalized flux")
            else:
                ax.set_ylabel("")
            if j == 0:
                ax.set_title(v["name"])
            ax.grid(True, alpha=0.3)
    fig.suptitle(f"{PLANET} — Flux vs systematics for all visits")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


### One subplot per visit: light curves only

In [ ]:
if visits:
    n = len(visits)
    nrow = min(1, n)
    ncol = (n + nrow - 1) // nrow
    fig, axes = plt.subplots(nrow, ncol, figsize=(8 * ncol, 5 * nrow), squeeze=False)
    axes = axes.flatten()
    for i, v in enumerate(visits):
        ax = axes[i]
        c = COLORS[i % len(COLORS)]
        ax.errorbar(v["time"], v["flux"], yerr=v["flux_err"], fmt=".", ms=3, alpha=0.9, color=c, capsize=1)
        ax.set_title(v["name"])
        ax.set_xlabel("BJD (days)")
        ax.set_ylabel("Normalized flux")
        #ax.set_ylim(0.995, 1.005)
        ax.grid(True, alpha=0.3)
        ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
    for j in range(len(visits), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(f"{PLANET} — Light curves by visit")
    fig.tight_layout()
    plt.show()

### Secondary eclipse fit for each visit

In [ ]:
import numpy as np
import dynesty
from dynesty import utils as dyfunc
import batman
import os
import pickle
import corner
import matplotlib.pyplot as plt

# pick first visit
v = visits[0]
result_base = f"dynesty_results/TOI1518b/dynesty_result_visit_{v['name']}"
os.makedirs("dynesty_results/TOI1518b", exist_ok=True)

# extract arrays
t = np.asarray(v["time"])
f = np.asarray(v["flux"])
fe = np.asarray(v["flux_err"])
roll = np.asarray(v["roll_angle"])
cx = np.asarray(v["centroid_x"])
cy = np.asarray(v["centroid_y"])
sm = np.asarray(v["smearing_lc"])
bc = np.asarray(v["background"])

# normalize systematics (zero-mean, unit std)
def norm(x):
    x = np.asarray(x, dtype=float)
    xm = np.nanmedian(x)
    xs = np.nanstd(x)
    if xs <= 0:
        return x - xm
    return (x - xm) / xs

roll_n = norm(roll)
cx_n = norm(cx)
cy_n = norm(cy)
sm_n = norm(sm)
bc_n = norm(bc)

# Known orbital parameters (from exoplanet archive))
t0_known = 2459983.791942     # Time of inferior conjunction (BJD)
P_known = 1.90261131          # days
rp_known = 0.09939            # Rp/R*
a_known = 4.109               # a/R*
inc_known = 77.626            # degrees
T14_known = 2.3950            # transit duration in hours 

# batman setup helper: build light curve for given eclipse parameters (t0, depth)
def batman_eclipse_model(tarr, t_secondary, fp, t0=t0_known, per=P_known, rp=rp_known, a=a_known, inc=inc_known):
    # period, a, inclination are fixed (obtained from exoplanet archive: most precise values)
    # we only fit t_secondary and fp here
    params = batman.TransitParams()        # object to store transit parameters
    params.t_secondary = t_secondary       # eclipse time
    params.fp = fp                         # planet-to-star flux ratio (eclipse depth)
    params.t0 = t0                         # Time of inferior conjunction (BJD)
    params.per = per                       # Orbital period (days)
    params.rp = rp                         # Planet-to-star radius ratio
    params.a = a                           # Semi-major axis (in units of stellar radii)
    params.inc = inc                       # Orbital inclination (degrees)
    params.ecc = 0.0                       # Eccentricity (0 for circular)
    params.w = 90.0                        # Argument of periastron (degrees)
    params.u = [0.0, 0.0]                  # Limb darkening coefficients
    params.limb_dark = "quadratic"
    m = batman.TransitModel(params, tarr, transittype="secondary")
    return m.light_curve(params)

# priors setup for common parameters
min_value = -0.01
max_value = 0.01
t_span = t.max() - t.min()
t_secondary_min, t_secondary_max = t.min() - 0.1 * t_span, t.max() + 0.1 * t_span
fp_min, fp_max = 0.0, 0.1
jitter_min, jitter_max = 0.5, 3.0
offset_min, offset_max = min_value, max_value
slope_min, slope_max = min_value, max_value
c_roll_sin_min, c_roll_sin_max = min_value, max_value
c_roll_cos_min, c_roll_cos_max = min_value, max_value
c_roll_sin2_min, c_roll_sin2_max = min_value, max_value
c_roll_cos2_min, c_roll_cos2_max = min_value, max_value
c_roll_sin3_min, c_roll_sin3_max = min_value, max_value
c_roll_cos3_min, c_roll_cos3_max = min_value, max_value
c_cy_min, c_cy_max = min_value, max_value
c_cx_min, c_cx_max = min_value, max_value
c_sm_min, c_sm_max = min_value, max_value
c_bc_min, c_bc_max = min_value, max_value

# parameter bounds for modular prior handling
param_bounds = {
    "t_secondary": (t_secondary_min, t_secondary_max),
    "fp": (fp_min, fp_max),
    "offset": (offset_min, offset_max),
    "slope": (slope_min, slope_max),
    "jitter": (jitter_min, jitter_max),
    "c_roll_sin": (c_roll_sin_min, c_roll_sin_max),
    "c_roll_cos": (c_roll_cos_min, c_roll_cos_max),
    "c_roll_sin2": (c_roll_sin2_min, c_roll_sin2_max),
    "c_roll_cos2": (c_roll_cos2_min, c_roll_cos2_max),
    "c_roll_sin3": (c_roll_sin3_min, c_roll_sin3_max),
    "c_roll_cos3": (c_roll_cos3_min, c_roll_cos3_max),
    "c_cx": (c_cx_min, c_cx_max),
    "c_cy": (c_cy_min, c_cy_max),
    "c_sm": (c_sm_min, c_sm_max),
    "c_bc": (c_bc_min, c_bc_max),
}

# make systematics modular
sys_vectors = {
    "c_cx": cx_n,
    "c_cy": cy_n,
    "c_sm": sm_n,
    "c_bc": bc_n,
}

def build_model_flux(theta, param_names, tarr):
    p = dict(zip(param_names, theta))
    bat = batman_eclipse_model(tarr, p["t_secondary"], p["fp"])

    # common additive correction
    corr = p["offset"]

    # add linear time slope term
    if "slope" in p:
        corr += p["slope"] * (tarr - np.nanmedian(tarr))

    # add linear systematics terms
    for name, vec in sys_vectors.items():
        if name in p:
            corr += p[name] * vec

    # apply simple additive roll-angle trig correction
    roll_rad = np.radians(roll)
    if "c_roll_sin" in p:
        corr += p["c_roll_sin"] * np.sin(roll_rad)
    if "c_roll_cos" in p:
        corr += p["c_roll_cos"] * np.cos(roll_rad)
    if "c_roll_sin2" in p:
        corr += p["c_roll_sin2"] * np.sin(2.0 * roll_rad)
    if "c_roll_cos2" in p:
        corr += p["c_roll_cos2"] * np.cos(2.0 * roll_rad)
    if "c_roll_sin3" in p:
        corr += p["c_roll_sin3"] * np.sin(3.0 * roll_rad)
    if "c_roll_cos3" in p:
        corr += p["c_roll_cos3"] * np.cos(3.0 * roll_rad)

    return bat + corr, bat, corr

def make_prior(param_names):
    def prior(unit):
        out = []
        for u, name in zip(unit, param_names):
            pmin, pmax = param_bounds[name]
            out.append(pmin + u * (pmax - pmin))
        return out
    return prior

def make_loglike(param_names):
    def loglike(theta):
        p = dict(zip(param_names, theta))
        m, _, _ = build_model_flux(theta, param_names, t)
        resid = (f - m)
        sigma = np.asarray(fe) * p["jitter"]
        invvar = 1.0 / (sigma ** 2)
        return -0.5 * np.sum(resid * resid * invvar + np.log(2 * np.pi * (sigma ** 2)))
    return loglike

def show_corner_plot(res, param_names, title):
    """Extract samples from result object and show a corner plot with the given title."""
    # corner plot
    try:
        samples = res.samples
        weights = np.exp(res.logwt - res.logz[-1])
        weights /= np.sum(weights)

        samples_equal = dyfunc.resample_equal(samples, weights)

        fig = corner.corner(samples_equal, labels=param_names, show_titles=True, title_fmt=".5f", quantiles=[0.16, 0.5, 0.84], 
                            title_kwargs={"fontsize": 14}, label_kwargs={"fontsize": 14}, tick_kwargs={"labelsize": 14}, max_n_ticks=4)

        # move the axis labels further from the tick labels
        for ax in fig.get_axes():
            # force label position farther from tick labels
            if ax.get_xlabel():
                ax.xaxis.set_label_coords(0.5, -0.5)
            if ax.get_ylabel():
                ax.yaxis.set_label_coords(-0.5, 0.5)
        
        # separate title names in 2 lines for better readability
        for ax in fig.get_axes():
            title = ax.get_title()
            if "=" in title:
                parts = title.split("=")
                new_title = "=\n".join(parts)
                ax.set_title(new_title, fontsize=14)

        plt.suptitle("Dynesty posterior samples for joint fit")
        plt.show()

    except Exception as e:
        print("Failed to produce corner plot:", e)

def run_dynesty_with_cache(model_name, param_names, loglike, prior, result_path):
    if os.path.exists(result_path):
        with open(result_path, "rb") as fh:
            res = pickle.load(fh)
        print(f"Loaded existing dynesty result from {result_path}")
    else:
        ndim = len(param_names)

        sampler = dynesty.NestedSampler(
            loglike,
            prior,
            ndim,
            nlive=400,
            bound='multi',
            sample='rwalk'
        )

        sampler.run_nested(dlogz=0.5)
        res = sampler.results

        try:
            with open(result_path, "xb") as fh:
                pickle.dump(res, fh)
                print(f"Saved dynesty result to {result_path}")
        except Exception as e:
            print(f"Could not save result to {result_path}: {e}")

    # show corner plot for this model
    show_corner_plot(
        res,
        param_names,
        f"Posterior samples: visit {v['name']} | model: {model_name}\nparams: {', '.join(param_names)}"
    )

    return res

def get_logz(res):
    return float(res.logz[-1])

def run_model_for_params(param_names, model_tag):
    loglike = make_loglike(param_names)
    prior = make_prior(param_names)

    # use descriptive file name: visit name + model tag
    result_path = f"{result_base}__{model_tag}.pkl"

    return run_dynesty_with_cache(model_tag, param_names, loglike, prior, result_path)

# candidate systematics to add one at a time; roll_angle expands to sin (and optionally cos)
systematic_candidates = [
    # each entry: (extra_param_names_to_add, tag_suffix)
    (["c_roll_sin"], "roll_sin"),
    (["c_roll_sin", "c_roll_cos"], "roll_sin__roll_cos"),
    (["c_roll_sin2"], "roll_sin2"),
    (["c_roll_sin2", "c_roll_cos2"], "roll_sin2___roll_cos2"),
    (["c_roll_sin3"], "roll_sin3"),
    (["c_roll_sin3", "c_roll_cos3"], "roll_sin3__roll_cos3"),
    (["c_cx"], "cx"),
    (["c_cy"], "cy"),
    (["c_sm"], "sm"),
    (["c_bc"], "bc"),
]

all_results = {}

# start from base model
current_param_names = ["t_secondary", "fp", "offset", "slope", "jitter"]
current_tag = "base"
current_result = run_model_for_params(current_param_names, current_tag)
current_logz = get_logz(current_result)
all_results[current_tag] = (current_param_names.copy(), current_result)

print(f"Base model logZ = {current_logz:.6f}")

# iterative forward selection: keep adding the single best-improving systematic
remaining_candidates = systematic_candidates.copy()

while remaining_candidates:

    best_improvement_logz = current_logz
    best_improvement_candidate = None
    best_improvement_result = None
    best_improvement_params = None
    best_improvement_tag = None

    # test all remaining candidates and pick the one that improves logZ the most
    for extra_params, suffix in remaining_candidates:

        # skip if any of the extra params are already in the current model
        if any(p in current_param_names for p in extra_params):
            continue

        trial_params = current_param_names + extra_params
        trial_tag = f"{current_tag}__{suffix}"

        trial_result = run_model_for_params(trial_params, trial_tag)
        trial_logz = get_logz(trial_result)

        all_results[trial_tag] = (trial_params.copy(), trial_result)

        print(f"  Tested {trial_tag}: logZ = {trial_logz:.6f} (current best: {best_improvement_logz:.6f})")

        if trial_logz > best_improvement_logz:
            best_improvement_logz = trial_logz
            best_improvement_candidate = (extra_params, suffix)
            best_improvement_result = trial_result
            best_improvement_params = trial_params
            best_improvement_tag = trial_tag

    if best_improvement_candidate is not None:
        # accept the best-improving candidate and update the current model
        current_param_names = best_improvement_params
        current_tag = best_improvement_tag
        current_result = best_improvement_result
        current_logz = best_improvement_logz

        remaining_candidates.remove(best_improvement_candidate)

        print(f"New best model '{current_tag}': logZ improved to {current_logz:.6f}")
        print(f"  Added params: {best_improvement_candidate[0]}")

    else:
        # no candidate improved the model; stop iterating
        print("No further improvement found. Stopping forward selection.")
        break

# final selected model summary
print("\n=== Final model summary ===")
print("Best model tag:", current_tag)
print("Best parameters:", current_param_names)
logzerr = current_result.logzerr[-1]
print(f"Best logZ: {current_logz:.6f} ± {logzerr:.6f}") 

try:
    samples = current_result.samples
    weights = np.exp(current_result.logwt - current_result.logz[-1])
    weights /= np.sum(weights)

    theta_mean = np.average(samples, axis=0, weights=weights)
    theta_error = np.sqrt(np.average((samples - theta_mean) ** 2, axis=0, weights=weights))

    print("Posterior-weighted mean parameters:")
    for n, val, err in zip(current_param_names, theta_mean, theta_error):
        print(f"  {n}: {val:.7f} ± {err:.7f}")

except Exception as e:
    print("Could not compute posterior-weighted means:", e)

In [ ]:
import numpy as np

def posterior_mean_std(result):
    """Return posterior-weighted mean and std from a dynesty result."""
    samples = np.asarray(result.samples)
    weights = np.exp(result.logwt - result.logz[-1])
    weights /= np.sum(weights)

    mean = np.average(samples, axis=0, weights=weights)
    var = np.average((samples - mean) ** 2, axis=0, weights=weights)
    std = np.sqrt(var)
    return mean, std


if "all_results" not in globals() or not all_results:
    print("No model results found in `all_results`.")
else:
    print("=== Model summaries (all models) ===")

    best_model_tag = None
    best_logz = -np.inf
    best_logzerr = np.nan
    best_param_names = None
    best_mean = None
    best_std = None

    for model_tag, (param_names, result) in all_results.items():
        print(f"\nFinal model summary ===")
        print(f"Model tag: {model_tag}")
        print(f"Parameters: {param_names}")

        if result is None:
            print("No fit result available.")
            continue

        try:
            logz = float(np.asarray(result.logz)[-1])
            logzerr = float(np.asarray(result.logzerr)[-1]) if hasattr(result, "logzerr") else np.nan
            mean, std = posterior_mean_std(result)

            print(f"logZ: {logz:.6f} ± {logzerr:.6f}")
            print("Posterior-weighted mean parameters:")
            for name, m, s in zip(param_names, mean, std):
                print(f"  {name}: {m:.7f} ± {s:.7f}")

            if logz > best_logz:
                best_model_tag = model_tag
                best_logz = logz
                best_logzerr = logzerr
                best_param_names = param_names
                best_mean = mean
                best_std = std

        except Exception as e:
            print(f"Could not summarize result: {e}")

    if best_model_tag is not None:
        print("\n=== Best model (by logZ) ===")
        print(f"Best model tag: {best_model_tag}")
        print(f"Best parameters: {best_param_names}")
        print(f"Best logZ: {best_logz:.6f} ± {best_logzerr:.6f}")
        print("Posterior-weighted mean parameters:")
        for name, m, s in zip(best_param_names, best_mean, best_std):
            print(f"  {name}: {m:.7f} ± {s:.7f}")


In [ ]:
# Extract mean parameters from fitted results
def extract_theta_mean(result, n_params):
    """Extract posterior mean from dynesty result."""
    if result is None:
        return None

    try:
        samples = np.asarray(result.samples)
        weights = np.exp(result.logwt - result.logz[-1])
        weights /= np.sum(weights)

        return np.average(samples, axis=0, weights=weights)

    except Exception:
        return None


# Extract mean parameters from fitted results and compare fitted vs. expected eclipse timing
print("=== Eclipse Timing Validation ===\n")

# Check all accumulated results from iterative model selection
for model_tag, (param_names, result) in all_results.items():
    print(f"\n{model_tag}:")
    print(f"  Parameters: {param_names}")

    if result is None:
        print("  No fit result available.")
        continue

    try:
        # Extract posterior samples + weights from dynesty result
        samples = np.asarray(result.samples)

        weights = np.exp(result.logwt - result.logz[-1])
        weights /= np.sum(weights)

    except Exception:
        print("  Could not read dynesty result.")
        continue

    # Extract posterior mean
    theta_mean = np.average(samples, axis=0, weights=weights)

    # Extract t_secondary (always first parameter in our setup)
    t_secondary_fit = theta_mean[0]

    # Use known orbital parameters
    t0 = t0_known
    P = P_known

    # Compute n that minimizes the difference
    n_float = ((t_secondary_fit - t0) / P) - 0.5
    n_nearest = int(np.round(n_float))

    t_secondary_expected = t0 + (n_nearest + 0.5) * P

    # Also check all t_secondary values in the time range
    t_min = min(t)
    t_max = max(t)

    n_start = int(np.ceil(((t_min - t0) / P) - 0.5))
    n_end = int(np.floor(((t_max - t0) / P) - 0.5))

    print(f"\nVisit {v['name']} time range: [{t_min:.8f}, {t_max:.8f}]")
    print(f"  t_secondary values within time range:")

    for n in range(n_start, n_end + 1):
        t_sec = t0 + (n + 0.5) * P
        print(f"    n={n}: t_secondary={t_sec:.8f}")

    print(f"  Fitted t_secondary:    {t_secondary_fit:.8f}")
    print(f"  Expected t_secondary:  {t_secondary_expected:.8f} (n={n_nearest})")
    print(f"  Difference:            {(t_secondary_fit - t_secondary_expected) * 24 * 60:.4f} minutes")

In [ ]:
# plot data with the fitted eclipse interval shaded

for model_tag, (param_names, result) in all_results.items():
    print(f"\n=== {model_tag} ===")

    if result is None:
        print("  No fit result available.")
        continue

    # Extract theta_mean from result
    try:
        samples = result.samples
        weights = np.exp(result.logwt - result.logz[-1])
        weights /= np.sum(weights)
        theta_mean = np.average(samples, axis=0, weights=weights)
    except Exception:
        print("  No fit result available.")
        continue

    if theta_mean is not None:
        # Use the modular build_model_flux function
        best_model, bat, corr = build_model_flux(theta_mean, param_names, t)

        # define a simple criterion for the eclipse region from the batman component (half of the fitted depth):
        fp = theta_mean[1]
        thresh = np.max(bat) - 0.5 * fp
        eclipse_mask = bat < thresh

        if np.any(eclipse_mask):
            t_start, t_end = t[eclipse_mask].min(), t[eclipse_mask].max()
        else:
            t_start = t_end = None

        # define a simple criterion for the eclipse region from the batman component (half of the fitted depth):
        p = dict(zip(param_names, theta_mean))
        fp = p.get("fp", np.nan)

        if np.isfinite(fp) and fp > 0:
            thresh = np.max(bat) - 0.5 * fp
            eclipse_mask = bat < thresh
        else:
            eclipse_mask = bat < np.max(bat)

        if np.any(eclipse_mask):
            t_start, t_end = t[eclipse_mask].min(), t[eclipse_mask].max()
        else:
            t_start = t_end = None

        fig, ax = plt.subplots(figsize=(13, 5))

        ax.errorbar(
            t, f, yerr=fe,
            fmt=".", ms=3, alpha=0.6,
            label="data"
        )

        ax.plot(
            t, best_model,
            color="C1", lw=1.5,
            label="best-fit model"
        )

        ax.plot(
            t, bat,
            "--", color="C2", lw=1.0,
            label="batman eclipse component"
        )

        if t_start is not None:
            ax.axvspan(
                t_start, t_end,
                color="gray", alpha=0.3,
                label="eclipse"
            )

            ax.axvline(
                t_start,
                color="gray", ls="--", alpha=0.5
            )

            ax.axvline(
                t_end,
                color="gray", ls="--", alpha=0.5
            )

        ax.set_xlabel("BJD (days)")
        ax.set_ylabel("Normalized flux")
        ax.grid(alpha=0.3)
        ax.legend(loc="best", fontsize=11)

        plt.title(f"{PLANET} visit {v['name']}\n{model_tag}")
        plt.show()

## Bayesian comparison

In [ ]:
# Load all new iterative-model results

result_dir = "dynesty_results/TOI1518b"
result_prefix = f"dynesty_result_visit_{v['name']}_iter__"
result_paths = {}

# Prefer the results already available from the iterative fitting cell
if "all_results" in globals() and all_results:
    for model_tag in all_results:
        result_paths[model_tag] = os.path.join(result_dir, f"{result_prefix}{model_tag}.pkl")
else:
    if os.path.isdir(result_dir):
        for fname in sorted(os.listdir(result_dir)):
            if fname.startswith(result_prefix) and fname.endswith(".pkl"):
                model_tag = fname[len(result_prefix):-4]
                result_paths[model_tag] = os.path.join(result_dir, fname)

results = {}
param_name_map = {}

if "all_results" in globals() and all_results:
    for model_tag, (param_names, res) in all_results.items():
        results[model_tag] = res
        param_name_map[model_tag] = param_names
        print(f"Using in-memory result for {model_tag}")
else:
    for name, path in result_paths.items():
        if os.path.exists(path):
            with open(path, "rb") as f:
                results[name] = pickle.load(f)
            print(f"Loaded {name} result from {path}")
        else:
            print(f"Warning: {path} not found")

# Extract log evidence (logZ) from each result
if len(results) > 1:
    logz_vals = {}
    logz_err_vals = {}
    rms_vals = {}

    for name, res in results.items():
        try:
            logz = float(res.logz[-1])
        except Exception:
            logz = None

        try:
            logz_err = float(res.logzerr[-1])
        except Exception:
            logz_err = np.nan

        logz_vals[name] = logz
        logz_err_vals[name] = logz_err

        try:
            if name in param_name_map:
                samples = np.asarray(res.samples)
                weights = np.exp(res.logwt - res.logz[-1])
                weights /= np.sum(weights)
                theta_mean = np.average(samples, axis=0, weights=weights)

                model_flux, _, _ = build_model_flux(theta_mean, param_name_map[name], t)
                rms = np.sqrt(np.mean((f - model_flux) ** 2))
            else:
                rms = np.nan
        except Exception:
            rms = np.nan

        rms_vals[name] = rms

    valid_logz = {name: logz for name, logz in logz_vals.items() if logz is not None}

    print("\n=== Model Comparison via Bayesian Evidence ===")

    if valid_logz:
        best_model = max(valid_logz, key=valid_logz.get)
        best_logz = valid_logz[best_model]

        for name, logz in sorted(valid_logz.items(), key=lambda x: x[1], reverse=True):
            logz_err = logz_err_vals.get(name, np.nan)
            rms = rms_vals.get(name, np.nan)
            print(f"{name:20s}: logZ = {logz:.3f} ± {logz_err:.3f}   delta_logZ = {logz - best_logz:.3f}   rms = {rms:.6f}")


    else:
        print("No valid logZ values found.")

else:
    print("Not enough results loaded for comparison")

import pandas as pd

# Ensure we have the logZ values available (recompute if necessary)
if "valid_logz" not in globals():
    # fall back to reloading the results dict if needed
    results = {}
    for name, path in result_paths.items():
        if os.path.exists(path):
            with open(path, "rb") as f:
                results[name] = pickle.load(f)

    valid_logz = {}
    logz_err_vals = {}
    rms_vals = {}

    for name, res in results.items():
        try:
            valid_logz[name] = float(res.logz[-1])
        except Exception:
            pass

        try:
            logz_err_vals[name] = float(res.logzerr[-1])
        except Exception:
            logz_err_vals[name] = np.nan

        try:
            if name in param_name_map:
                samples = np.asarray(res.samples)
                weights = np.exp(res.logwt - res.logz[-1])
                weights /= np.sum(weights)
                theta_mean = np.average(samples, axis=0, weights=weights)

                model_flux, _, _ = build_model_flux(theta_mean, param_name_map[name], t)
                rms_vals[name] = np.sqrt(np.mean((f - model_flux) ** 2))
            else:
                rms_vals[name] = np.nan
        except Exception:
            rms_vals[name] = np.nan

# Build a table
df = pd.DataFrame(
    [(name, logz, logz_err_vals.get(name, np.nan), rms_vals.get(name, np.nan)) for name, logz in valid_logz.items()],
    columns=["model", "logZ", "logZ_err", "rms"]
)
df = df.sort_values("logZ", ascending=False).reset_index(drop=True)

best_logz = df.loc[0, "logZ"]

name_best = df.loc[0, "model"]

# Get uncertainty on logZ for the best model
try:
    best_logz_err = float(results[name_best].logzerr[-1])
except Exception:
    best_logz_err = float("nan")

print(f"\nBest model: {name_best} (logZ = {best_logz:.3f} ± {best_logz_err:.3f})")

df["delta_logZ"] = df["logZ"] - best_logz

# Put rms at the end
df = df[["model", "logZ", "logZ_err", "delta_logZ", "rms"]]

df


### RMS–binning / Allan deviation test for residuals

In [ ]:
# Allan deviation-style RMS vs binsize for residuals of each fitted model

if not visits:
    print("No visits loaded.")
elif "all_results" not in globals() or not all_results:
    print("No iterative model results available in all_results.")
else:
    # Helper: weighted posterior mean parameter vector
    def posterior_mean_theta(res):
        if res is None: 
            return None
        try:
            samples = np.asarray(res.samples)
            weights = np.exp(res.logwt - res.logz[-1]); weights /= np.sum(weights)
            return np.average(samples, axis=0, weights=weights)
        except Exception:
            return None

    # Helper: RMS of residuals after binning by contiguous bins of size m
    def binned_rms(y, m):
        nbin = len(y) // m
        if nbin < 2: return np.nan
        y_trim = y[:nbin * m]; y_bin = y_trim.reshape(nbin, m).mean(axis=1)
        return np.sqrt(np.mean((y_bin - np.mean(y_bin)) ** 2))

    # cadence in minutes (for secondary x-axis)
    cadence_min = np.nanmedian(np.diff(np.asarray(t))) * 24 * 60

    model_info = [(model_tag, param_names, result) for model_tag, (param_names, result) in all_results.items()]
    nrows = len(model_info) // 2 + len(model_info) % 2
    fig, axes = plt.subplots(nrows, 2, figsize=(12, 3*nrows), squeeze=False)
    axes = axes.flatten()

    for i, (name, param_names, res) in enumerate(model_info):
        ax = axes[i]
        theta = posterior_mean_theta(res)

        if theta is None:
            ax.set_title(f"{name}\n(no samples)"); ax.axis("off"); continue

        # Build model flux
        try:
            model_flux, bat, corr = build_model_flux(theta, param_names, t)
        except Exception:
            ax.set_title(f"{name}\n(model build failed)"); ax.axis("off"); continue

        if "f" in globals() and len(np.asarray(f)) == len(np.asarray(model_flux)):
            flux_data = np.asarray(f, dtype=float)
        else:
            flux_data = np.asarray(v["flux"], dtype=float)

        model_flux = np.asarray(model_flux, dtype=float)
        resid = flux_data - model_flux
        good = np.isfinite(resid); resid = resid[good]

        # Use powers-of-two bins for a clean Allan/RMS curve
        max_m = max(2, len(resid) // 10)
        binsizes = []; m = 1
        while m <= max_m:
            binsizes.append(m); m *= 2
        binsizes = np.array(binsizes, dtype=int)

        rms_vals = np.array([binned_rms(resid, m) for m in binsizes], dtype=float)
        valid = np.isfinite(rms_vals) & (rms_vals > 0)

        if np.any(valid):
            rms_vals_ppm = rms_vals[valid] * 1e6
            ax.loglog(binsizes[valid], rms_vals_ppm, "o-", ms=4, label="Residual RMS")

            # White-noise reference: RMS(1)/sqrt(m)
            rms1 = rms_vals[binsizes == 1][0] if np.any(binsizes == 1) else rms_vals[valid][0]
            ref = rms1 / np.sqrt(binsizes[valid])
            ax.loglog(binsizes[valid], ref * 1e6, "--", lw=1.2, label=r"$\propto N^{-1/2}$")

            # top x-axis: bin size converted to minutes
            axt = ax.twiny()
            axt.set_xscale("log")
            axt.set_xlim(ax.get_xlim())
            axt.set_xticks(binsizes[valid])
            axt.set_xticklabels([f"{(bs * cadence_min):.1f}" for bs in binsizes[valid]])
            axt.set_xlabel("Bin size (minutes)")

        if "name_best" in globals() and name == name_best: 
            ax.set_title(f"Model:{name} (best model)", color="red")
        else: 
            ax.set_title(f"Model:{name}")
        ax.set_xlabel("Bin size (points)")
        ax.set_ylabel("RMS of binned residuals (ppm)")
        #ax.set_ylim(bottom=40)
        #ax.set_ylim(top=500)
        ax.set_yticks([60, 100, 200, 300, 400, 500])
        ax.set_yticklabels(["60", "100", "200", "300", "400", "500"])
        ax.grid(True, which="both", alpha=0.3)
        ax.legend()


    # hide any unused axes
    for j in range(len(model_info), len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"{PLANET} visit {v['name']} — Allan deviation / RMS-binning test")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
# Allan deviation-style RMS vs binsize for residuals of each fitted model 

if not visits:
    print("No visits loaded.")
elif "all_results" not in globals() or not all_results:
    print("No iterative model results available in all_results.")
else:
    # Helper: weighted posterior mean parameter vector
    def posterior_mean_theta(res):
        if res is None: 
            return None
        try:
            samples = np.asarray(res.samples)
            weights = np.exp(res.logwt - res.logz[-1]); weights /= np.sum(weights)
            return np.average(samples, axis=0, weights=weights)
        except Exception:
            return None

    # Helper: RMS of residuals after binning by contiguous bins of size m
    def binned_rms(y, m):
        nbin = len(y) // m
        if nbin < 2: return np.nan
        y_trim = y[:nbin * m]; y_bin = y_trim.reshape(nbin, m).mean(axis=1)
        return np.sqrt(np.mean((y_bin - np.mean(y_bin)) ** 2))

    # cadence in minutes (for secondary x-axis)
    cadence_min = np.nanmedian(np.diff(np.asarray(t))) * 24 * 60

    # keep only the models with delta logZ < 1 from the best model
    scored_models = []
    for model_tag, (param_names, result) in all_results.items():
        try:
            logz_val = float(np.asarray(result.logz)[-1])
        except Exception:
            logz_val = -np.inf
        scored_models.append((logz_val, model_tag, param_names, result))

    scored_models = sorted(scored_models, key=lambda x: x[0], reverse=True)
    best_logz = scored_models[0][0] if scored_models else -np.inf
    scored_models = [x for x in scored_models if np.isfinite(x[0]) and (best_logz - x[0] < 1.0)]

    if not scored_models:
        print("No models satisfy delta logZ < 1.")
    else:
        model_info = [(model_tag, param_names, result) for _, model_tag, param_names, result in scored_models]
        nrows = len(model_info) // 2 + len(model_info) % 2
        fig, axes = plt.subplots(nrows, 2, figsize=(14, 6*nrows), squeeze=False)
        axes = axes.flatten()

        for i, (name, param_names, res) in enumerate(model_info):
            ax = axes[i]
            theta = posterior_mean_theta(res)

            if theta is None:
                ax.set_title(f"{name}\n(no samples)"); ax.axis("off"); continue

            # Build model flux
            try:
                model_flux, bat, corr = build_model_flux(theta, param_names, t)
            except Exception:
                ax.set_title(f"{name}\n(model build failed)"); ax.axis("off"); continue

            if "f" in globals() and len(np.asarray(f)) == len(np.asarray(model_flux)):
                flux_data = np.asarray(f, dtype=float)
            else:
                flux_data = np.asarray(v["flux"], dtype=float)

            model_flux = np.asarray(model_flux, dtype=float)
            resid = flux_data - model_flux
            good = np.isfinite(resid); resid = resid[good]

            # Use powers-of-two bins for a clean Allan/RMS curve
            max_m = max(2, len(resid) // 10)
            binsizes = []; m = 1
            while m <= max_m:
                binsizes.append(m); m *= 2
            binsizes = np.array(binsizes, dtype=int)

            rms_vals = np.array([binned_rms(resid, m) for m in binsizes], dtype=float)
            valid = np.isfinite(rms_vals) & (rms_vals > 0)

            if np.any(valid):
                rms_vals_ppm = rms_vals[valid] * 1e6
                ax.loglog(binsizes[valid], rms_vals_ppm, "o-", ms=4, label="Residual RMS")

                # White-noise reference: RMS(1)/sqrt(m)
                rms1 = rms_vals[binsizes == 1][0] if np.any(binsizes == 1) else rms_vals[valid][0]
                ref = rms1 / np.sqrt(binsizes[valid])
                ax.loglog(binsizes[valid], ref * 1e6, "--", lw=1.2, label=r"$\propto N^{-1/2}$")

                # top x-axis: bin size converted to minutes
                axt = ax.twiny()
                axt.set_xscale("log")
                axt.set_xlim(ax.get_xlim())
                axt.set_xticks(binsizes[valid])
                axt.set_xticklabels([f"{(bs * cadence_min):.1f}" for bs in binsizes[valid]])
                axt.set_xlabel("Bin size (minutes)")

            ax.set_title(f"{name} \nlogZ={float(np.asarray(res.logz)[-1]):.3f}")
            ax.set_xlabel("Bin size (points)")
            ax.set_ylabel("RMS of binned residuals (ppm)")
            #ax.set_ylim(bottom=40)
            #ax.set_ylim(top=500)
            ax.set_yticks([60, 100, 200, 300, 400, 500])
            ax.set_yticklabels(["60", "100", "200", "300", "400", "500"])
            ax.grid(True, which="both", alpha=0.3)
            ax.legend()

        # hide any unused axes
        for j in range(len(model_info), len(axes)):
            axes[j].axis("off")

        fig.suptitle(f"{PLANET} visit {v['name']} — Allan deviation / RMS-binning test")
        fig.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()

In [ ]:
# Residuals vs systematics for each fitted model
if not visits:
    print("No visits loaded.")
elif "all_results" not in globals() or not all_results:
    print("No iterative model results available in all_results.")
else:
    flux_data, time_data = np.asarray(v["flux"], dtype=float), np.asarray(v["time"], dtype=float)
    systematics = [("time", time_data), ("roll_angle", np.asarray(v["roll_angle"], dtype=float)),
                   ("centroid_x", np.asarray(v["centroid_x"], dtype=float)), ("centroid_y", np.asarray(v["centroid_y"], dtype=float)),
                   ("smearing_lc", np.asarray(v["smearing_lc"], dtype=float)), ("background", np.asarray(v["background"], dtype=float))]

    def posterior_mean_theta(res):
        if res is None: return None
        try:
            samples = np.asarray(res.samples); weights = np.exp(res.logwt - res.logz[-1]); weights /= np.sum(weights)
            return np.average(samples, axis=0, weights=weights)
        except Exception:
            return None

    model_specs = [(model_tag, param_names, result) for model_tag, (param_names, result) in all_results.items()]
    n_rows, n_cols = len(model_specs), len(systematics)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 2.8 * n_rows), squeeze=False)

    for i, (model_name, param_names, res) in enumerate(model_specs):
        theta = posterior_mean_theta(res)
        if theta is None:
            for j in range(n_cols):
                axes[i, j].text(0.5, 0.5, "No samples", ha="center", va="center"); axes[i, j].set_axis_off()
            continue

        try:
            model_flux, bat, corr = build_model_flux(theta, param_names, time_data); model_flux = np.asarray(model_flux, dtype=float)
        except Exception:
            for j in range(n_cols):
                axes[i, j].text(0.5, 0.5, "Model build failed", ha="center", va="center"); axes[i, j].set_axis_off()
            continue

        n = min(len(flux_data), len(model_flux)); resid = flux_data[:n] - model_flux[:n]
        for j, (sys_name, sys_vals) in enumerate(systematics):
            ax = axes[i, j]; x = np.asarray(sys_vals[:n], dtype=float); good = np.isfinite(x) & np.isfinite(resid)
            ax.plot(x[good], resid[good], ".", ms=3, alpha=0.6, color=COLORS[i % len(COLORS)])
            ax.axhline(0.0, color="gray", ls="--", lw=1, alpha=0.7); ax.grid(True, alpha=0.25)
            if i == 0: ax.set_title(sys_name)
            if j == 0: ax.set_ylabel(f"{model_name}\nresidual")
            else: ax.set_ylabel("")
            if i == n_rows - 1: ax.set_xlabel(sys_name)

    fig.suptitle(f"{PLANET} visit {v['name']} — Residuals vs systematics by model")
    fig.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()